In [1]:
import pandas as pd
import sqlite3
import os

project_path = os.path.join(os.path.expanduser("~"), "Documents", "Healthcare_SQL_Analytics")
data_path = os.path.join(project_path, "data", "hospital_readmissions.csv")
db_path = os.path.join(project_path, "healthcare_readmissions.db")

df = pd.read_csv(data_path)

df = df.reset_index(drop=True)
df.insert(0, "encounter_id", df.index + 1)   # wristband number: 1..25000

print("Rows:", len(df))
df[["encounter_id", "age", "medical_specialty", "readmitted"]].head()

Rows: 25000


,encounter_id,age,medical_specialty,readmitted
0,1,[70-80),Missing,no
1,2,[70-80),Other,no
2,3,[50-60),Missing,yes
3,4,[70-80),Missing,yes
4,5,[60-70),InternalMedicine,no


In [2]:
dim_age = pd.DataFrame({"age_band": sorted(df["age"].unique())})
dim_age["age_min"] = dim_age["age_band"].str.extract(r"\[(\d+)-").astype(int)
dim_age["age_max"] = dim_age["age_band"].str.extract(r"-(\d+)\)").astype(int)
dim_age = dim_age.sort_values("age_min").reset_index(drop=True)
dim_age["sort_order"] = dim_age.index + 1
dim_age

,age_band,age_min,age_max,sort_order
0,[40-50),40,50,1
1,[50-60),50,60,2
2,[60-70),60,70,3
3,[70-80),70,80,4
4,[80-90),80,90,5
5,[90-100),90,100,6


In [3]:
dim_specialty = pd.DataFrame({"specialty_name": sorted(df["medical_specialty"].unique())})
dim_specialty.insert(0, "specialty_id", dim_specialty.index + 1)
dim_specialty

,specialty_id,specialty_name
0,1,Cardiology
1,2,Emergency/Trauma
2,3,Family/GeneralPractice
3,4,InternalMedicine
4,5,Missing
5,6,Other
6,7,Surgery


In [4]:
diag_values = pd.unique(df[["diag_1", "diag_2", "diag_3"]].values.ravel())
diag_values = sorted([d for d in diag_values if pd.notna(d)])
dim_diagnosis = pd.DataFrame({"diagnosis_category": diag_values})
dim_diagnosis.insert(0, "diagnosis_id", dim_diagnosis.index + 1)
dim_diagnosis

,diagnosis_id,diagnosis_category
0,1,Circulatory
1,2,Diabetes
2,3,Digestive
3,4,Injury
4,5,Missing
5,6,Musculoskeletal
6,7,Other
7,8,Respiratory


In [5]:
diag_long = df.melt(
    id_vars="encounter_id",
    value_vars=["diag_1", "diag_2", "diag_3"],
    var_name="diag_col",
    value_name="diagnosis_category"
)
diag_long["diagnosis_position"] = diag_long["diag_col"].str.replace("diag_", "").astype(int)
diag_long = diag_long.merge(dim_diagnosis, on="diagnosis_category", how="left")

bridge = (diag_long[["encounter_id", "diagnosis_id", "diagnosis_position"]]
          .sort_values(["encounter_id", "diagnosis_position"])
          .reset_index(drop=True))

print("Bridge rows (should be 25000 × 3 = 75000):", len(bridge))
bridge.head(6)

Bridge rows (should be 25000 × 3 = 75000): 75000


,encounter_id,diagnosis_id,diagnosis_position
0,1,1,1
1,1,8,2
2,1,7,3
3,2,7,1
4,2,7,2
5,2,7,3


In [6]:
fact = df.merge(dim_specialty, left_on="medical_specialty", right_on="specialty_name", how="left")

fact_encounters = fact[[
    "encounter_id", "age", "specialty_id",
    "time_in_hospital", "n_lab_procedures", "n_procedures", "n_medications",
    "n_outpatient", "n_inpatient", "n_emergency",
    "glucose_test", "A1Ctest", "change", "diabetes_med", "readmitted"
]].rename(columns={"age": "age_band", "A1Ctest": "a1c_test", "change": "med_change"})

fact_encounters.head()

,encounter_id,age_band,specialty_id,time_in_hospital,n_lab_procedures,n_procedures,n_medications,n_outpatient,n_inpatient,n_emergency,glucose_test,a1c_test,med_change,diabetes_med,readmitted
0,1,[70-80),5,8,72,1,18,2,0,0,no,no,no,yes,no
1,2,[70-80),6,3,34,2,13,0,0,0,no,no,no,yes,no
2,3,[50-60),5,5,45,0,18,0,0,0,no,no,yes,yes,yes
3,4,[70-80),5,2,36,0,12,1,0,0,no,no,yes,yes,yes
4,5,[60-70),4,1,42,0,7,0,0,0,no,no,no,yes,no


In [7]:
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON;")

conn.executescript("""
DROP TABLE IF EXISTS bridge_encounter_diagnosis;
DROP TABLE IF EXISTS fact_encounters;
DROP TABLE IF EXISTS dim_age;
DROP TABLE IF EXISTS dim_specialty;
DROP TABLE IF EXISTS dim_diagnosis;

CREATE TABLE dim_age (
    age_band   TEXT PRIMARY KEY,
    age_min    INTEGER,
    age_max    INTEGER,
    sort_order INTEGER
);

CREATE TABLE dim_specialty (
    specialty_id   INTEGER PRIMARY KEY,
    specialty_name TEXT NOT NULL
);

CREATE TABLE dim_diagnosis (
    diagnosis_id       INTEGER PRIMARY KEY,
    diagnosis_category TEXT NOT NULL
);

CREATE TABLE fact_encounters (
    encounter_id     INTEGER PRIMARY KEY,
    age_band         TEXT,
    specialty_id     INTEGER,
    time_in_hospital INTEGER,
    n_lab_procedures INTEGER,
    n_procedures     INTEGER,
    n_medications    INTEGER,
    n_outpatient     INTEGER,
    n_inpatient      INTEGER,
    n_emergency      INTEGER,
    glucose_test     TEXT,
    a1c_test         TEXT,
    med_change       TEXT,
    diabetes_med     TEXT,
    readmitted       TEXT,
    FOREIGN KEY (age_band)     REFERENCES dim_age (age_band),
    FOREIGN KEY (specialty_id) REFERENCES dim_specialty (specialty_id)
);

CREATE TABLE bridge_encounter_diagnosis (
    encounter_id       INTEGER,
    diagnosis_id       INTEGER,
    diagnosis_position INTEGER,
    PRIMARY KEY (encounter_id, diagnosis_position),
    FOREIGN KEY (encounter_id) REFERENCES fact_encounters (encounter_id),
    FOREIGN KEY (diagnosis_id) REFERENCES dim_diagnosis (diagnosis_id)
);
""")
conn.commit()

# load parents first, children last
dim_age.to_sql("dim_age", conn, if_exists="append", index=False)
dim_specialty.to_sql("dim_specialty", conn, if_exists="append", index=False)
dim_diagnosis.to_sql("dim_diagnosis", conn, if_exists="append", index=False)
fact_encounters.to_sql("fact_encounters", conn, if_exists="append", index=False)
bridge.to_sql("bridge_encounter_diagnosis", conn, if_exists="append", index=False)
conn.commit()

print("Tables created and loaded.")

Tables created and loaded.


In [8]:
print(pd.read_sql("SELECT COUNT(*) AS encounters FROM fact_encounters;", conn))
print(pd.read_sql("SELECT COUNT(*) AS diagnosis_rows FROM bridge_encounter_diagnosis;", conn))

pd.read_sql("""
SELECT a.age_band,
       COUNT(*) AS encounters,
       ROUND(AVG(CASE WHEN f.readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters f
JOIN dim_age a ON f.age_band = a.age_band
GROUP BY a.age_band, a.sort_order
ORDER BY a.sort_order;
""", conn)

   encounters
0       25000
   diagnosis_rows
0           75000


,age_band,encounters,readmit_pct
0,[40-50),2532,44.5
1,[50-60),4452,44.2
2,[60-70),5913,46.8
3,[70-80),6837,48.8
4,[80-90),4516,49.6
5,[90-100),750,42.1
